# 🎭 Teddy Face Swap — GPU (free test)

**Just 2 taps, then wait ~4 minutes:**

1. Top menu → **Runtime → Change runtime type → T4 GPU → Save**  *(may already be set)*
2. Top menu → **Runtime → Run all**

When it finishes, a **https link** appears at the bottom of cell **6**. **Tap it on your phone** → *Start Camera* → smooth face swap on a real GPU.

If anything is wrong, a cell **STOPS with a message that starts with `STOP:`** — copy that whole output to the agent and it will be fixed. Nothing keeps running in a broken/CPU-only state.

⚠️ Keep this Colab tab open while you use it — if it closes, the server stops (free-tier limit).

In [ ]:
# 1/7  Check we actually got a GPU
import subprocess
o = subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],capture_output=True,text=True)
if o.returncode or not o.stdout.strip():
    raise SystemExit('STOP: no GPU. Fix: Runtime > Change runtime type > T4 GPU > Save, then Runtime > Run all again.')
print('GPU:', o.stdout.strip())

In [ ]:
# 2/7  Get the app code
import os, subprocess
REPO='https://github.com/oluwacoded/Deepfakcall'; BRANCH='main'
if not os.path.isdir('/content/Deepfakcall'):
    if subprocess.run(['git','clone','--depth','1','-b',BRANCH,REPO,'/content/Deepfakcall']).returncode:
        raise SystemExit('STOP: clone failed. Check the connection and send the agent this output.')
os.chdir('/content/Deepfakcall')
if not (os.path.exists('web_server.py') and os.path.exists('templates/index.html')):
    raise SystemExit('STOP: code missing after clone. Send the agent this output.')
print('Code ready in', os.getcwd())

In [ ]:
# 3/7  Install GPU dependencies (remove conflicting builds first) + tunnel tool
import subprocess, sys, os
# drop CPU onnxruntime and any mixed OpenCV wheels so we end up with one clean GPU stack
subprocess.run([sys.executable,'-m','pip','-q','uninstall','-y',
                'onnxruntime','onnxruntime-gpu',
                'opencv-python','opencv-contrib-python','opencv-python-headless'])
if subprocess.run([sys.executable,'-m','pip','-q','install','-r','requirements-gpu.txt']).returncode:
    raise SystemExit('STOP: pip install failed. Send the agent this whole output.')
if subprocess.run('wget -q -O /tmp/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /tmp/cloudflared', shell=True).returncode or not os.path.exists('/tmp/cloudflared'):
    raise SystemExit('STOP: cloudflared download failed. Send the agent this output.')
print('dependencies installed OK')

In [ ]:
# 4/7  Confirm the GPU is usable by the AI runtime (STOPS if not — this is what makes it fast)
import onnxruntime as ort
p = ort.get_available_providers(); print('providers:', p)
if 'CUDAExecutionProvider' not in p:
    raise SystemExit('STOP: onnxruntime cannot see the GPU (no CUDAExecutionProvider). '
                     'On CPU it would be slow. Copy this whole output to the agent to fix the CUDA/onnxruntime version.')
print('GPU ready for AI runtime')

In [ ]:
# 5/7  Pre-download the Amber Song face model (~718 MB) to this GPU box
import os, subprocess
os.makedirs('userdata/models', exist_ok=True)
m='userdata/models/Amber_Song.dfm'
url='https://github.com/iperov/DeepFaceLive/releases/download/AMBER_SONG/Amber_Song.dfm'
MIN=700_000_000
ok=lambda: os.path.exists(m) and os.path.getsize(m)>=MIN
for attempt in range(1,4):
    if ok(): break
    print('downloading model, attempt', attempt, flush=True)
    subprocess.run(['wget','-q','--show-progress','-c','-O',m,url])
if not ok():
    raise SystemExit('STOP: model download incomplete ('+str(os.path.getsize(m) if os.path.exists(m) else 0)+' bytes). Re-run this cell; if it keeps failing tell the agent.')
print('model ready:', round(os.path.getsize(m)/1e6), 'MB')

In [ ]:
# 6/7  Start the app, wait until it is actually healthy, then show a public link
import subprocess, time, re, os, sys, urllib.request
os.environ['SESSION_SECRET']='colab-'+os.urandom(8).hex()
def tail(f,n=15):
    try: return ''.join(open(f).readlines()[-n:])
    except Exception: return '(no '+f+' yet)'
srv=subprocess.Popen([sys.executable,'web_server.py'],stdout=open('server.log','w'),stderr=subprocess.STDOUT)
print('starting server...', flush=True)
ready=False
for _ in range(60):
    time.sleep(2)
    if srv.poll() is not None:
        print('SERVER LOG:\n'+tail('server.log',25)); raise SystemExit('STOP: server crashed on startup. Send the agent the SERVER LOG above.')
    try:
        urllib.request.urlopen('http://127.0.0.1:5000/api/models', timeout=3); ready=True; break
    except Exception: pass
if not ready:
    print('SERVER LOG:\n'+tail('server.log',25)); raise SystemExit('STOP: server did not become ready. Send the agent the SERVER LOG above.')
print('server is healthy', flush=True)
tun=subprocess.Popen(['/tmp/cloudflared','tunnel','--url','http://localhost:5000','--no-autoupdate'],stdout=open('tunnel.log','w'),stderr=subprocess.STDOUT)
url=None
for _ in range(45):
    time.sleep(2)
    mm=re.search(r'https://[-a-z0-9]+\.trycloudflare\.com', tail('tunnel.log',200))
    if mm: url=mm.group(0); break
print('\n'+'='*58)
if url:
    print('OPEN THIS ON YOUR PHONE (tap it):\n\n    '+url+'\n')
    print('  1) tap Start Camera, allow the camera')
    print('  2) Amber Song is preloaded — tap Load Model')
    print('  3) keep this Colab tab open while you use it')
else:
    print('TUNNEL LOG:\n'+tail('tunnel.log',25))
    print('Tunnel not ready — send the agent the TUNNEL LOG above.')
print('='*58)

In [ ]:
# 7/7  Keeps the session awake + shows live activity. Leave this running.
import time
while True:
    time.sleep(30)
    try: print(''.join(open('server.log').readlines()[-6:]), flush=True)
    except Exception as e: print('log:', e)